# SupplementRx AI — KG Retrieval and Evaluation

**Status: Step 1 question/criteria draft complete; Q01–Q20 have assistant source-relevance annotations pending user adjudication; Q21–Q30, retrieval implementation and scoring remain pending.** The KG and frozen releases are unchanged.

## Broad steps

1. Define 20–30 representative questions and expected-result criteria across all ten supplements.
2. Review original source evidence and annotate expected results, exclusions and answerability. Check warning/safety coverage here, separating source gaps from export/link/retrieval gaps.
3. Build a simple graph-query baseline that returns evidence, citations, mapping relation/confidence and source review status.
4. Evaluate precision, recall, appropriate abstention and preservation of source roles/qualifications. Investigate failures before embeddings or GNNs.

The 30 draft questions below use a pinned graph version. Evidence seeds are convenient records to inspect, **not a reviewed gold set**. Source trust and extraction correctness are distinct: pending NatMed annotations remain attributed statements, while their original text/context is reviewed for relevance.

**Latest revision:** collection-wide statement preservation fixes the demonstrated warning export gap. See the final section for the current graph and evaluation bindings; earlier outputs describe preserved historical versions.

**Latest evaluation revision:** see Step 2C at the end for NatMed relevance decisions and `retrieval_questions_source_review_v3.json`. Earlier sections preserve historical stages.

**Latest evaluation status:** Q01–Q30 have assistant-reviewed development expectations. A fixed-query baseline and first precision/recall evaluation are complete; see the final sections for results and limitations. Earlier sections preserve historical stages.

**Latest relevance experiment:** v2 improves original-set scores but loses recall on 12 additional diagnostic cases. V1 remains the default baseline; see the final comparison section.


In [1]:
from pathlib import Path
import json, hashlib
root = Path.cwd()
if not (root / 'data').is_dir():
    root = root.parent
question_file = root / 'data/evaluation/retrieval_questions_v1.json'
evaluation = json.loads(question_file.read_text())
graph = root / evaluation['graph']
assert hashlib.sha256((graph / 'manifest.json').read_bytes()).hexdigest() == evaluation['graph_manifest_sha256']
questions = evaluation['questions']
assert len(questions) == 30 and len({q['id'] for q in questions}) == 30
assert len({q['topic'] for q in questions}) == 10
assert all(q['expected_relevant_evidence_ids'] is None for q in questions)
print('Pinned graph:', evaluation['graph'])
print('Questions:', len(questions), '| Topics:', len({q['topic'] for q in questions}))
print('Status:', evaluation['status'])


Pinned graph: data/graph/full-graph-navigation-v1-073a086cbc98af6632ae
Questions: 30 | Topics: 10
Status: Question and criteria draft; gold evidence annotations and retrieval baseline not implemented


## Step 1 — Questions and expected-result criteria

Each topic has one product-warning question, one NatMed question and one contrasting retrieval/limitation case. Product IDs identify saved labels, not necessarily currently marketed products. The question file stores criteria, candidate paths, evidence seeds, exclusions and abstention conditions.

**Warning field inspection:** DSLD has 2,023 `LabelAssertion` nodes of kind `label_statement`; one inspected record contains “Do not use if inner safety seal is broken or missing.” This confirms a storage route, not complete warning coverage. Statement type/text must be reviewed to distinguish warnings from other label content. Warnings should be retrieved through `ProductLabel → HAS_ASSERTION`, not assumed present in the label snapshot properties.

NatMed's saved claim types include effectiveness (809), interaction (148), adverse_effect (120), safety (107) and dose (85). These are annotation counts, not verified completeness or clinical approval.


In [2]:
for q in questions:
    print(f"{q['id']} | {q['topic']} | {q['category']}")
    print(q['question'])
    for criterion in q['expected_result_criteria']:
        print('  -', criterion)
    print()


Q01 | Magnesium | product_label_warnings
What warnings and usage restrictions are stated on Quad-Magnesium (DSLD label 242165)?
  - Inspect every source label statement; select actual warnings/restrictions without treating every statement as a warning.
  - Return original wording, product/label ID, source and relevant conditions.
  - Separate label directions from restrictions and distinguish no saved warning from evidence of no risk.

Q02 | Vitamin D | product_label_warnings
What warnings and usage restrictions are stated on Vitamin D (DSLD label 239369)?
  - Inspect every source label statement; select actual warnings/restrictions without treating every statement as a warning.
  - Return original wording, product/label ID, source and relevant conditions.
  - Separate label directions from restrictions and distinguish no saved warning from evidence of no risk.

Q03 | Ashwagandha | product_label_warnings
What warnings and usage restrictions are stated on Ashwagandha Extra Strength (DSL

## Step 2 — Annotate expected source evidence (partially completed)

For each question, inspect the original raw label or monograph section—not just the example graph records. Record relevant source IDs/spans, qualifications, exclusions and expected answerability. Map these to graph evidence only after source review. Null expected-result fields currently mean **not reviewed**, not “no relevant results.”

Classify a warning/safety gap as:

| Gap | Meaning |
|---|---|
| Source missing | Needed information is absent from the saved source capture; wider source completeness is unknown. |
| Collected, not exported | Raw information exists but is absent from graph representations. |
| Exported, not linked | Information exists in the graph but the required supported path is absent. |
| Retrieval miss | A supported path exists but the baseline fails to return it. |

Preserve source roles: manufacturer label warnings, NatMed ingredient statements and Reddit discussion are separate. A name/topic mention never establishes verified composition, strain equivalence or product-specific clinical applicability. Missing warnings do not establish safety.


In [3]:
# Inspection aid only; this does not construct a gold set or run a retriever.
example = questions[0]
print(json.dumps({k: example[k] for k in ['id', 'question', 'evidence_seed_node_ids', 'candidate_graph_paths', 'abstention_conditions', 'expected_relevant_evidence_ids', 'annotation_status']}, indent=2))
print()
print('Source-review protocol:')
for item in evaluation['annotation_protocol']:
    print('-', item)


{
  "id": "Q01",
  "question": "What warnings and usage restrictions are stated on Quad-Magnesium (DSLD label 242165)?",
  "evidence_seed_node_ids": [
    "snapshot:4a1b00d83635591e4d514b2e80eb6c860c3d5e81a76a78d2889e91231efda776",
    "assertion:e56da1920c5578fdd117119b1cfd793bb6c9045d726b16c846731f41fc0b0475",
    "assertion:6df3778755d4c0c78fc0995e9a508296274f6ae03625d730dfd568198c70512b"
  ],
  "candidate_graph_paths": [
    "ProductLabel \u2192 HAS_ASSERTION \u2192 LabelAssertion(kind=label_statement)"
  ],
  "abstention_conditions": [
    "If no warning is captured, report that coverage gap; do not invent warnings."
  ],
  "expected_relevant_evidence_ids": null,
  "annotation_status": "draft_pending_source_review"
}

Source-review protocol:
- Review raw DSLD label statements and NatMed source sections, including relevant context and qualifications.
- Record relevant source spans and source IDs first; then record whether each is preserved/exported/linked in the graph.
- Label gaps

### Step 2A — Source review of ten DSLD warning questions

**Q01–Q10 have assistant source-relevance annotations, pending user adjudication.** This is an assessment of what the saved label says, not clinical approval. Q11–Q30 remain unannotated. The original draft question file is preserved; annotations are in `data/evaluation/retrieval_questions_source_review_v1.json`.

Reviewed all **216 statements** across the ten selected raw labels before matching them to graph assertions. Explicit include/exclude decisions and rationales are saved in `dsld_warning_review_decisions_v1.json`; reproducible source/hash/path validation is in `scripts/review_warning_evidence.py`.

**Evaluation scope:** product warnings, consultation requirements, allergen notices, and usage restrictions. Routine dosing directions, storage, regulatory disclaimers and marketing are not primary warning evidence. When directions contain a restriction, retain the complete original statement as contextual evidence. Do not convert its dosage text into a recommendation. The warnings apply to the full labeled product, including co-ingredients—not automatically to the topic ingredient alone.

**Duplicates:** identical statement text within one label is one relevant evaluation unit with alternative source pointers/assertion IDs. Partial overlaps remain separate. The expected evidence-ID lists are alternatives grouped by unit, not a requirement to retrieve every repeated copy.


In [4]:
import sys
if str(root / 'scripts') not in sys.path:
    sys.path.insert(0, str(root / 'scripts'))
from review_warning_evidence import run
warning_review, reviewed_questions = run(root)
print(f'{"Question":9} {"Topic":14} {"Label":8} {"Reviewed":>8} {"Included":>8} {"Units":>6} {"Missing":>8}')
for row in warning_review['questions']:
    missing = row['included_coverage'].get('collected_not_exported', 0) + row['included_coverage'].get('exported_not_linked', 0)
    print(f'{row["question_id"]:9} {row["topic"]:14} {row["label_id"]:8} {row["statements_reviewed"]:8} {row["included_source_statements"]:8} {row["unique_warning_units"]:6} {missing:8}')


{
  "questions_reviewed": 10,
  "statements_reviewed": 216,
  "included_source_statements": 66,
  "unique_warning_units": 60,
  "included_coverage": {
    "preserved_and_linked": 64,
    "collected_not_exported": 2
  }
}
Question  Topic          Label    Reviewed Included  Units  Missing
Q01       Magnesium      242165         14        6      5        0
Q02       Vitamin D      239369         20        6      4        0
Q03       Ashwagandha    206077         18        6      5        0
Q04       Melatonin      3749           24        9      9        0
Q05       Omega-3        223186         19        7      7        0
Q06       Vitamin B12    179549         22        5      5        0
Q07       Iron           214846         23        7      7        0
Q08       Turmeric       179934         19        6      5        0
Q09       Probiotics     63275          33        9      8        0
Q10       Creatine       25731          24        5      5        2


In [5]:
# Change to Q01–Q10 to inspect source decisions for another selected label.
warning_question_id = 'Q10'
for row in warning_review['statement_decisions']:
    if row['question_id'] != warning_question_id:
        continue
    print(row['source_pointer'], '|', row['decision'], '|', row['coverage_state'])
    print('Source type:', row['original_statement']['type'])
    print('Original text:', row['original_statement']['notes'])
    print('Reason:', row['rationale'])
    print('Graph bindings:', row['graph_bindings'])
    print()


/statements/0 | exclude | collected_not_exported
Source type: General: Product/Version Code
Original text: Rev. 01-001-CPS001 03/13
Reason: Product description, marketing, composition/free-from claim, identity, symbol or administrative content; not a warning/use restriction in this question scope.
Graph bindings: []

/statements/1 | exclude | collected_not_exported
Source type: Brand IP Statement(s) re: (c), (TM), (SM)
Original text: {chart}
These POWERFUL Stacks when used together are designed to help promote the ultimate Anabolic physique.
WORKOUT DAYS
STACKING PROTOCOL PRE-WORKOUT POST-WORKOUT NON-WORKOUT DAYS MORNING AFTERNOON
BETA-ALKALINE(TM) {scoop} x 1 30 - 45 MIN PRIOR {image} x 1
CREATINE-ALKALINE(TM) {image} x2 IMMEDIATELY AFTER {image} x 2
GLUTA-ALKALINE(TM) (image} x 1 1 - 2 HOURS PRIOR {image} x 1
LEUCINE AGMA pH(TM) {image} x 1 {image} x 1

Reason: Product description, marketing, composition/free-from claim, identity, symbol or administrative content; not a warning/use r

### Findings and required follow-up

- **66 included source-statement occurrences → 60 distinct warning-text units.** All duplicates retain their source locations; no graph records were merged.
- **64 included occurrences are preserved and linked.** Two are collected but not exported, both on Creatine label **25731**: `/statements/18` (“Please read entire label before use.”) and `/statements/21` (“Do not exceed recommended dose.”). These remain required source evidence, even though their graph binding lists are empty.
- **Cause:** `scripts/preprocess_pilot.py` currently selects statement types containing `direction`, `precaution` or `storage`. Both missing restrictions are typed as general content. This is an export-selection issue, not a retrieval algorithm failure.
- **False positive to avoid:** Iron label **214846**, `/statements/3`, is typed as a precaution but describes formulation/absorption benefits. It is excluded by content review. Its adverse-event reporting contact is optional support, not a primary warning.
- **Restrictions inside directions:** retained for Melatonin, Vitamin B12, Iron and Probiotics. Filtering only precaution types would miss relevant context.
- **Product versus ingredient:** the selected Probiotics product includes warnings about iron and vitamin A; the selected Vitamin D product includes a vitamin K2/anticoagulant precaution. Retain these as whole-product label statements; do not relabel them as intrinsic probiotic or vitamin D warnings.

No evidence of additional warnings was found in a keyword screen of other raw string fields for these ten labels. That screen is not proof of absence, and this review does not establish completeness of source images, uncaptured labels or all 400 labels.

**Status:** source-relevance annotation is assistant-reviewed, awaiting user adjudication. The graph, raw data and frozen releases are unchanged. No retrieval precision/recall has been measured. Before scoring, address the demonstrated export gap with a general collection-wide statement-preservation rule, then continue expected-evidence review for the NatMed and contrasting questions.


## Step 3 — Graph-query baseline (not started)

Implement question-specific queries after reviewing expected results. Return source record IDs, excerpts, citations/URLs, actual path edges, confidence scoped to its relationship, and review status. Separate direct Reddit mentions from thread context. A missing path or insufficient evidence must be returned explicitly.

## Step 4 — Evaluation (criteria defined; not run)

- **Precision:** relevant retrieved evidence units divided by all returned evidence units.
- **Recall:** relevant retrieved units divided by all adjudicated relevant units in the saved-source scope.
- Define units per source (label assertion/span, clinical claim/span, Reddit record), deduplicate repeated returns, and preserve distinct contexts.
- Empty returned sets have undefined precision; empty relevant sets have undefined recall. Report N/A and evaluate abstention separately rather than assigning artificial perfect scores.
- Report results per question and source before aggregation. Check attribution, qualifications, null values, pending review, identity scope and context separation alongside retrieval metrics.
- These 30 questions are an initial development set, not an independent benchmark. If later training/tuning uses them, create a separate held-out set grouped to prevent leakage across related labels, excerpts and threads.

**Next action:** address the documented general-content export gap, then review the remaining NatMed and contrasting questions against raw source evidence. No retrieval scores, clinical approvals, embeddings or model training have been produced here.


### Statement-preservation revision — collection-wide fix

**Current graph:** `data/graph/full-graph-statements-v1-27c544f4940e7162944c`  
**Frozen revision:** `data/graph/poc-statements-release-v1-684ce1012b4f76e0af1a`

All **6,462 source statements across 400 captured DSLD labels** are now preserved and connected to their product labels. The previous graph retained 2,022 statements-array entries; this revision adds 4,440 omitted entries. (The older 2,023 label-statement count also included one other-ingredient text assertion.)

The rule is **preserve all captured statement types**, not “classify every statement as a warning.” Original types and wording remain unchanged, including marketing, disclaimers, allergen information and general-content restrictions. Warning relevance remains a separate content-review decision. No source/clinical corrections, identity merges or mapping promotions were made.

The revision has **35,998 nodes and 74,088 relationships**. Existing nodes, relationships, sources and frozen versions remain intact. Full captured-statement coverage is not a claim of label-image completeness or clinical correctness.

Reproduce with `scripts/preserve_dsld_statements.py`; policy: `schemas/dsld_statement_preservation_v1.json`.


In [6]:
from pathlib import Path
import json, hashlib
statement_root = Path.cwd()
if not (statement_root / 'data').is_dir():
    statement_root = statement_root.parent
statement_graph = statement_root / 'data/graph/full-graph-statements-v1-27c544f4940e7162944c'
statement_release = statement_root / 'data/graph/poc-statements-release-v1-684ce1012b4f76e0af1a'
statement_freeze = json.loads((statement_release / 'freeze_manifest.json').read_text())
assert hashlib.sha256((statement_graph / 'manifest.json').read_bytes()).hexdigest() == statement_freeze['graph_manifest_sha256']
statement_report = json.loads((statement_graph / 'validation_report.json').read_text())
print(json.dumps({k: statement_report[k] for k in ['labels','source_statement_occurrences','previously_preserved','added_statements','nodes','edges','checks']}, indent=2))


{
  "labels": 400,
  "source_statement_occurrences": 6462,
  "previously_preserved": 2022,
  "added_statements": 4440,
  "nodes": 35998,
  "edges": 74088,
  "checks": {
    "all_captured_statements_preserved": true,
    "all_statement_paths_reachable": true,
    "existing_assertions_and_edges_unchanged": true,
    "no_clinical_approval": true,
    "no_mapping_changes": true,
    "source_values_and_hashes_verified": true
  }
}


### Updated warning bindings and continued NatMed inventory

The original graph-bound question draft and warning review remain historical artifacts. Use **`data/evaluation/retrieval_questions_source_review_v2.json`** for the new graph binding. Relevance decisions are unchanged: all **66 included occurrences / 60 distinct warning-text units** now resolve to graph assertions and `HAS_ASSERTION` paths. The two Creatine general-content restrictions are included; promotional text remains excluded from the warning evaluation.

Q11–Q20 now have a **source-first inventory** in `data/evaluation/natmed_question_evidence_inventory_v1.json`. It selects by raw monograph ID and requested annotation type, verifies original source excerpts and graph payload equality, and records candidate evidence with preserved dose/population/interaction fields. This is an inventory for review, not an adjudicated relevance set. Clinical approval remains false; expected gold evidence fields for these questions remain unfilled.


In [7]:
import sys
if str(statement_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(statement_root / 'scripts'))
from review_warning_evidence import run as review_warning_bindings
warning_review_v2, current_questions = review_warning_bindings(statement_root, draft_path='data/evaluation/retrieval_questions_statements_v1.json', output_version='v2')
assert warning_review_v2['summary']['included_coverage'] == {'preserved_and_linked': 66}
from inventory_natmed_evaluation import run as inventory_natmed_questions
natmed_inventory = inventory_natmed_questions(statement_root)
for row in natmed_inventory['question_summaries']:
    print(row['question_id'], row['topic'], row['requested_claim_type'], row['source_annotation_count'], row['sections'], row['coverage'])


{
  "questions_reviewed": 10,
  "statements_reviewed": 216,
  "included_source_statements": 66,
  "unique_warning_units": 60,
  "included_coverage": {
    "preserved_and_linked": 66
  }
}
{
  "questions": 10,
  "source_claim_candidates": 260,
  "coverage": {
    "preserved_and_linked": 260
  }
}
Q11 Magnesium interaction 24 {'interactions with drugs': 17, 'interactions with supplements': 5, 'interactions with lab tests': 2} {'preserved_and_linked': 24}
Q12 Vitamin D dose 18 {'dosing & administration': 18} {'preserved_and_linked': 18}
Q13 Ashwagandha safety 7 {'safety': 3, 'interactions with conditions': 4} {'preserved_and_linked': 7}
Q14 Melatonin safety 12 {'safety': 7, 'interactions with conditions': 5} {'preserved_and_linked': 12}
Q15 Omega-3 interaction 13 {'interactions with drugs': 9, 'interactions with supplements': 4} {'preserved_and_linked': 13}
Q16 Vitamin B12 effectiveness 62 {'effectiveness': 62} {'preserved_and_linked': 62}
Q17 Iron adverse_effect 13 {'adverse effects': 13

**Current evaluation status:** Q01–Q10 have assistant source-relevance annotations awaiting user adjudication, with the export gap fixed. Q11–Q20 have 260 verified source-annotation candidates, all preserved and linked; semantic relevance/context review remains pending. Q21–Q30 remain unannotated.

The NatMed inventory includes laboratory-test and supplement interactions as well as drug interactions. Safety candidates also occur in interactions-with-conditions sections; a section-heading-only query would be insufficient. Review complete source context and field anchors before turning candidates into expected results.

**Next:** finish the NatMed relevance/context review, then the contrasting cases. No general retrieval implementation, precision/recall scores or clinical approvals have been produced.


### Step 2C — NatMed relevance and context decisions for Q11–Q20

**Assistant source-relevance review complete for these ten captured-annotation questions; user adjudication and clinical review remain separate.** The graph and raw data are unchanged. Use `data/evaluation/retrieval_questions_source_review_v3.json` for the latest expected evidence (Q01–Q20); Q21–Q30 remain unannotated.

The review includes original claim fields, source excerpt/section context, supported ratings and field anchors. Policy: `data/evaluation/natmed_relevance_policy_v1.json`; reproducible verification: `scripts/review_natmed_relevance.py`; claim-level decisions: `data/evaluation/natmed_relevance_review_v1.json`.

- **258 primary units:** relevant to the requested saved annotation type and source monograph.
- **2 supporting context units:** Vitamin D's general statement that dose recommendations vary, and Iron's general tolerability statement. Return optionally and label as context; do not count them as numeric-dose answers or adverse-event examples.
- Other annotation types in the same monograph are listed as exclusions from each question's primary evidence set. This is task relevance, not a judgment that they are clinically irrelevant in general.
- Unit = source claim ID and its structured meaning/support. The 18 Vitamin D annotations share two source blocks but contain different dose/population/caution contexts. They must not be collapsed into two claims or treated as 18 independent studies.
- All 260 excerpts and 699 field-support anchors are verified. **129 anchors occur in earlier section headings**; keep their source pointers and text with the answer.


In [8]:
from pathlib import Path
import sys, json
natmed_review_root = Path.cwd()
if not (natmed_review_root / 'data').is_dir():
    natmed_review_root = natmed_review_root.parent
if str(natmed_review_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(natmed_review_root / 'scripts'))
from review_natmed_relevance import run as run_natmed_relevance
natmed_relevance_review, evaluation_questions_v3 = run_natmed_relevance(natmed_review_root)
print(f'{"ID":5} {"Topic":13} {"Actual source title":27} {"Primary":>7} {"Context":>7} {"Other-type exclusions":>22}')
for row in natmed_relevance_review['questions']:
    print(f'{row["question_id"]:5} {row["topic"]:13} {row["source_title"]:27} {row["primary_units"]:7} {row["supporting_context_units"]:7} {row["excluded_other_type_claims"]:22}')


{
  "questions_reviewed": 10,
  "candidate_annotations_reviewed": 260,
  "primary_units": 258,
  "supporting_context_units": 2,
  "source_excerpts_verified": 260,
  "field_support_anchors_verified": 699,
  "earlier_section_heading_anchors": 129,
  "shared_excerpt_groups": 2,
  "clinical_approvals": 0
}
ID    Topic         Actual source title         Primary Context  Other-type exclusions
Q11   Magnesium     Magnesium                        24       0                    144
Q12   Vitamin D     Vitamin D                        17       1                    175
Q13   Ashwagandha   Ashwagandha                       7       0                     66
Q14   Melatonin     Melatonin                        12       0                    147
Q15   Omega-3       Fish Oil                         13       0                    164
Q16   Vitamin B12   Vitamin B12                      62       0                     34
Q17   Iron          Iron                             12       1                     76


In [9]:
# Inspect one question's source scope and mandatory answer qualifications.
review_question_id = 'Q19'  # Change to Q11–Q20.
review_question = next(q for q in evaluation_questions_v3['questions'] if q['id'] == review_question_id)
print(review_question['question'])
print('Actual source monograph:', review_question['source_monograph_title'])
print('Primary source claim IDs:', len(review_question['expected_primary_claim_ids']))
for qualification in review_question['required_qualifications']:
    print('-', qualification)
example_unit = next(d for d in natmed_relevance_review['decisions'] if d['question_id'] == review_question_id and d['decision'] == 'primary')
print('Example source excerpt (attributed source text, not a recommendation):')
print(example_unit['original_excerpt'])
print('Verified field support:')
for anchor in example_unit['field_support_anchors']:
    print(anchor['field'], anchor['anchor_scope'], repr(anchor['source_text']), anchor['source_span'])


What indications and effectiveness ratings appear in the saved NatMed annotations for Probiotics?
Actual source monograph: Lactobacillus acidophilus
Primary source claim IDs: 75
- Display source title Lactobacillus acidophilus. This is not complete clinical evidence for all probiotics.
- Retain strain, live/heat-killed status, oral/intravaginal route, population, product and combination context in each excerpt.
- Include insufficient and negative evidence; preserve earlier-section rating anchors.
- Do not promote combination evidence to an individual strain, all species, or a branded probiotic product.
Example source excerpt (attributed source text, not a recommendation):
POSSIBLY EFFECTIVE
Antibiotic-associated diarrhea.
Most research shows that oral Lactobacillus acidophilus, alone or in combination with other probiotics, seems to reduce the risk of antibiotic-associated diarrhea (AAD) in adults.
A meta-analysis of 18 clinical trials in hospitalized or outpatient adults shows that ta

### Findings that must constrain retrieval and answers

| Questions | Required distinction |
|---|---|
| Q11 Magnesium | Drug interactions, supplement interactions and two laboratory-test interferences are separate source subtypes. |
| Q12 Vitamin D | Keep age/population, units, route, RDA/AI, upper limits and cautions separate. Unknown duration/frequency stays unknown; no new dose recommendation or conversion. |
| Q13/Q14/Q20 safety | General safety and condition-specific precautions are both relevant. Keep mixed routes, pregnancy/child contexts, short-/long-term qualifications and uncertainty. |
| Q15 Omega-3 | Actual captured source is **Fish Oil**, not all omega-3 materials or all products. |
| Q16 Vitamin B12 | Include positive, negative and insufficient-evidence ratings. Preserve non-oral routes and combination-only evidence. |
| Q17 Iron | Distinguish oral from intravenous/parenteral findings and association/case reports from causal conclusions. General tolerability is context. |
| Q18 Turmeric | Preserve theoretical, in-vitro/animal, formulation and conflicting-evidence qualifiers; do not turn every entry into a proven clinical event. |
| Q19 Probiotics | Actual captured source is **Lactobacillus acidophilus**. Preserve strain, live/heat-killed, route, population and combination qualifiers; do not generalize to all probiotics. |

**Status:** Q01–Q20 have assistant-reviewed expected evidence, pending user adjudication. The clinical annotations themselves remain pending; no graph updates, clinical approvals or precision/recall scores were produced. The reference scope is the saved annotation collection, not exhaustive monograph extraction or all clinical literature.

**Next:** annotate Q21–Q30 (quantity/mapping comparisons, product applicability, Reddit direct/context separation, null preservation and abstention), then implement the baseline retriever and score against reviewed units.


### Step 2D — Contrasting questions Q21–Q30

All 30 questions now have assistant-reviewed **development expectations**, pending user adjudication. The latest question file is `data/evaluation/retrieval_questions_source_review_v4.json`; scoring units and source/preparation provenance are in `data/evaluation/evaluation_cases_v1.json`. For mixed-unit questions, use `expected_primary_units` in the case file; legacy evidence-ID/span fields may remain unset intentionally.

| Case | Reviewed scope |
|---|---|
| Q21 Magnesium | Declared quantity, original units/operator/serving basis and comparison gate. |
| Q22 Vitamin D | High mappings for the Vitamin D occurrence and descendant forms; exclude Vitamin K co-ingredient mappings. |
| Q23 Ashwagandha | Label warnings, approved subject path and seven source safety annotations; no exact-product applicability inferred. |
| Q24 Melatonin | 135 direct Reddit records and 624 inherited-context records scored separately; no label-specific interpretation. |
| Q25 Omega-3 | Two reviewed ALA category links across the collection; no exact-material or example-product equivalence. |
| Q26 Vitamin B12 | Preserve dose/population nulls in the selected source annotation. |
| Q27 Iron | Label warnings versus the 14 saved safety/condition annotations, kept in separate source lanes. |
| Q28 Turmeric | Abstain on personal safety with an unspecified medicine and personal context. |
| Q29 Probiotics | Forty unique labels with explicit title/name mentions; no strain, viability or formulation equivalence. |
| Q30 Creatine | Compare the two declared amounts/bases and preserve ingredient differences; abstain on comparative effectiveness. |

Q24/Q25/Q29 were clarified as collection-level questions; the old example-label suffix was removed. Q26 now names the selected annotation explicitly. These scope clarifications are preserved in the new version, not silently applied to the old draft.


## Step 3 — Fixed graph-query baseline v1

Implementation: `scripts/kg_retrieval_baseline.py`. It accepts only a request (category, topic and selected source IDs); it never receives expected evidence IDs or reviewed relevance decisions. It follows graph edges and returns attributed evidence packets, source IDs/excerpts, mapping relations, original fields, source titles, support anchors and review status. It uses fixed abstention behavior for unsupported conclusions.

**This is not a general natural-language answering system.** Known entity IDs and question categories are supplied. No language-model-generated medical answers, ranking, top-k search, embeddings or GNN training are involved.

The initial warning classifier deliberately uses only the source `Precautions` type. It has no per-question exceptions copied from the review set. NatMed queries initially return all requested claim-type annotations as primary evidence, exposing primary/context errors. These are measured baseline limitations, not source export gaps.


In [10]:
from pathlib import Path
import sys, json
baseline_root = Path.cwd()
if not (baseline_root / 'data').is_dir():
    baseline_root = baseline_root.parent
if str(baseline_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(baseline_root / 'scripts'))
from annotate_contrasting_questions import run as annotate_contrasting
baseline_cases, baseline_questions = annotate_contrasting(baseline_root)
from evaluate_kg_retrieval import run as evaluate_baseline
baseline_report, baseline_responses = evaluate_baseline(baseline_root)


Annotated 30 development cases; Q21-Q30 primary sizes: {'Q21': 1, 'Q22': 4, 'Q23': 13, 'Q24': 135, 'Q25': 2, 'Q26': 1, 'Q27': 21, 'Q28': 0, 'Q29': 40, 'Q30': 2}
{
  "questions": 30,
  "macro_precision": 0.9836762854004233,
  "macro_recall": 0.9646004378762999,
  "questions_with_retrieval_errors": [
    "Q04",
    "Q06",
    "Q07",
    "Q09",
    "Q10",
    "Q12",
    "Q17",
    "Q27"
  ],
  "contract_violation_count": 0
}


## Step 4 — Development evaluation v1

Scoring compares unique primary evidence units against source/preparation-based reviewed expectations. Repeated returns receive no extra credit. Context has a separate role: optional context presented as primary is a primary-role false positive. Q24's inherited-context set has a separate score. Q28 has no relevant evidence that establishes personal safety; precision/recall are N/A and abstention is checked instead.

Macro precision/recall average the **29 questions with defined denominators**. This set was used to design the queries and some preparation rules are shared with the graph. Scores are development/conformance results—not independent clinical precision/recall, natural-language performance or a held-out benchmark.


In [11]:
def pct(value):
    return 'N/A' if value is None else f'{100*value:.1f}%'
print('Macro precision:', pct(baseline_report['macro_precision']))
print('Macro recall:', pct(baseline_report['macro_recall']))
print('Questions with retrieval errors:', baseline_report['questions_with_retrieval_errors'])
print('Structured contract violations:', baseline_report['contract_violation_count'])
print()
print(f'{"Question":9} {"Expected":>8} {"Returned":>8} {"FP":>4} {"FN":>4} {"Precision":>10} {"Recall":>9}')
for row in baseline_report['results']:
    s = row['scores']
    print(f'{row["question_id"]:9} {s["expected"]:8} {s["retrieved"]:8} {s["false_positive"]:4} {s["false_negative"]:4} {pct(s["precision"]):>10} {pct(s["recall"]):>9}')
print()
print('Per-source results:', json.dumps(baseline_report['by_source'], indent=2))


Macro precision: 98.4%
Macro recall: 96.5%
Questions with retrieval errors: ['Q04', 'Q06', 'Q07', 'Q09', 'Q10', 'Q12', 'Q17', 'Q27']
Structured contract violations: 0

Question  Expected Returned   FP   FN  Precision    Recall
Q01              5        5    0    0     100.0%    100.0%
Q02              4        4    0    0     100.0%    100.0%
Q03              5        5    0    0     100.0%    100.0%
Q04              9        8    0    1     100.0%     88.9%
Q05              7        7    0    0     100.0%    100.0%
Q06              5        4    0    1     100.0%     80.0%
Q07              7        8    2    1      75.0%     85.7%
Q08              5        5    0    0     100.0%    100.0%
Q09              8        7    0    1     100.0%     87.5%
Q10              5        3    0    2     100.0%     60.0%
Q11             24       24    0    0     100.0%    100.0%
Q12             17       18    1    0      94.4%    100.0%
Q13              7        7    0    0     100.0%    100.0%
Q14   

In [12]:
# Inspect any returned evidence packet and its scored mistakes.
inspect_question = 'Q10'  # Q24 shows direct/context separation; Q28/Q30 show abstention.
response = next(r for r in baseline_responses if r['question_id'] == inspect_question)
result = next(r for r in baseline_report['results'] if r['question_id'] == inspect_question)
print(json.dumps(result, indent=2))
print('Response status:', response['status'])
print('Limitations:', response['limitations'])
if response['primary']:
    print('First evidence packet:', json.dumps(response['primary'][0], indent=2))
print('Full responses: data/evaluation/baseline_v1/responses.json')


{
  "question_id": "Q10",
  "category": "product_label_warnings",
  "scores": {
    "expected": 5,
    "retrieved": 3,
    "true_positive": 3,
    "false_positive": 0,
    "false_negative": 2,
    "precision": 1.0,
    "recall": 0.6,
    "false_positive_ids": [],
    "false_negative_ids": [
      "warning:25731:29139f7e0fae84dcfd22354532b6042031e82d94645bd3a224668c2928bf2937",
      "warning:25731:d5dad743d634d553ff3738886e10d3a3d104ee3ae761315b10a2d0431c8b95d4"
    ]
  },
  "separate_context_scores": null,
  "contract_violations": [],
  "status": "source_evidence_only"
}
Response status: source_evidence_only
Limitations: ['Captured source evidence only; no independent clinical approval or personal recommendation.']
First evidence packet: {
  "unit_id": "warning:25731:e9af67f3af30d539049cd3228b7a03ec2ad3fee3b8aca5eeceb2bd55d23837d9",
  "source": "DSLD",
  "node_id": "assertion:2194d413e397cac775ae619e8cf926275445a9b6db60485ed224e7f4949c8052",
  "evidence_ids": [
    "evidence:7f54e0d63

### Baseline findings and next action

- **98.4% macro precision; 96.5% macro recall**, over the 29 questions with defined evidence denominators. Eight questions have retrieval errors: Q04, Q06, Q07, Q09, Q10, Q12, Q17 and Q27.
- The type-only warning filter misses restrictions in directions/general content (including the two repaired Creatine statements). It also retrieves Iron formulation marketing and a reporting contact as primary warnings. The graph now contains the missing evidence; the remaining issue is retrieval relevance.
- Vitamin D's general variability statement and Iron's general tolerability statement are returned as primary evidence instead of optional context.
- Q24 returns all 135 expected direct Reddit records and separately all 624 inherited-context records. Q28 abstains; Q30 compares declared amounts while abstaining on product superiority.
- Structured packet checks find no source-field, null, mapping-scope, context-promotion or unsupported-conclusion violations. Negative tests deliberately corrupt those fields to verify detection. These checks do not establish the safety of generated prose or clinical validity.
- Graph/raw data and frozen releases remain unchanged. Artifacts include full evidence responses, question-level errors, per-source metrics and hashes/code provenance under `data/evaluation/baseline_v1/`.

**Next:** improve general warning-relevance and primary/context classification, evaluate that revision against this development baseline, and add unseen examples before claiming generalization. Embeddings/GNNs are not needed to fix these demonstrated rule-based relevance failures.


## Relevance revision v2 — locked rules and additional diagnostic cases

**Experimental revision; v1 remains the default baseline.** `scripts/kg_retrieval_v2.py` adds content-based warning/use-restriction cues, separates reporting contacts/routine directions, and separates general NatMed context from primary evidence. It contains no evaluation IDs or expected relevance sets. The v1 script and results are preserved.

Rules were hashed in `data/evaluation/retrieval_v2_rule_lock.json` **before selecting and annotating the additional examples**. The comparison verifies the lock; v2 was not tuned after seeing its diagnostic failures.

### Additional cases and overlap controls

- Ten new warning labels (one per supplement), chosen by deterministic hash order, excluding original labels and connected source relationships/exact brand-product families. Selection does not use classifier outputs or warning text.
- Two additional NatMed questions: Ashwagandha adverse effects and Creatine dose contexts, using claim IDs outside the original evaluation set.
- Two selected labels contain no captured warning statements. Do not interpret this as evidence of safety.
- Of 24 primary warning occurrences, **5 have exact text already present in the original warning set and 19 have novel text**. Boilerplate overlap is disclosed rather than hidden.
- This is the same saved collection and same assistant reviewer, with relevance decisions pending user adjudication. It is a diagnostic expansion, **not an independent held-out benchmark**. After inspection, these cases become development data for future revisions.

Source decisions: `data/evaluation/additional_evaluation_cases_v1.json`; selection record: `additional_case_selection_v1.json`. Reproduce annotation/selection using `scripts/annotate_additional_evaluation_cases.py` and `scripts/select_additional_warning_cases.py` if the pinned inputs are unchanged.


In [13]:
from pathlib import Path
import sys, json
revision_root = Path.cwd()
if not (revision_root / 'data').is_dir():
    revision_root = revision_root.parent
if str(revision_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(revision_root / 'scripts'))
from compare_retrieval_revisions import run as compare_retrieval_revisions
revision_comparison = compare_retrieval_revisions(revision_root)


{
  "original_development": {
    "v1": {
      "macro_precision": 0.9836762854004233,
      "macro_recall": 0.9646004378762999,
      "precision_denominator": 29,
      "recall_denominator": 29,
      "errors": [
        "Q04",
        "Q06",
        "Q07",
        "Q09",
        "Q10",
        "Q12",
        "Q17",
        "Q27"
      ],
      "total_false_positives": 6,
      "total_false_negatives": 7,
      "contract_violations": 0
    },
    "v2": {
      "macro_precision": 1.0,
      "macro_recall": 0.9979716024340771,
      "precision_denominator": 29,
      "recall_denominator": 29,
      "errors": [
        "Q12"
      ],
      "total_false_positives": 0,
      "total_false_negatives": 1,
      "contract_violations": 0
    }
  },
  "additional_diagnostic": {
    "v1": {
      "macro_precision": 0.9722222222222221,
      "macro_recall": 0.975,
      "precision_denominator": 10,
      "recall_denominator": 10,
      "errors": [
        "H04",
        "H10",
        "H11"
      

In [14]:
print(f'{"Set / version":30} {"Precision":>10} {"Recall":>10} {"P denom":>8} {"R denom":>8} {"FP":>5} {"FN":>5}')
for set_name, versions in revision_comparison['sets'].items():
    for version, score in versions.items():
        print(f'{set_name + " / " + version:30} {100*score["macro_precision"]:9.1f}% {100*score["macro_recall"]:9.1f}% {score["precision_denominator"]:8} {score["recall_denominator"]:8} {score["total_false_positives"]:5} {score["total_false_negatives"]:5}')
print('Adoption decision:', revision_comparison['adoption_decision'])
print('Additional-example overlap:', revision_comparison['additional_overlap'])
print('No-warning cases have undefined precision/recall if no evidence is returned; they are not assigned perfect scores.')


Set / version                   Precision     Recall  P denom  R denom    FP    FN
original_development / v1           98.4%      96.5%       29       29     6     7
original_development / v2          100.0%      99.8%       29       29     0     1
additional_diagnostic / v1          97.2%      97.5%       10       10     2     1
additional_diagnostic / v2          97.8%      83.0%        9       10     1     4
Adoption decision: do_not_replace_v1_with_v2
Additional-example overlap: {'warning_occurrences_exact_text_overlap': 5, 'warning_occurrences_novel_text': 19, 'clinical_claim_id_overlap': 0}
No-warning cases have undefined precision/recall if no evidence is returned; they are not assigned perfect scores.


In [15]:
additional = json.loads((revision_root / 'data/evaluation/additional_evaluation_cases_v1.json').read_text())
by_unit = {d['unit_id']: d for d in additional['decisions']}
print('Additional v2 mistakes (original source text):')
for result in revision_comparison['sets']['additional_diagnostic']['v2']['results']:
    for kind in ['false_negative_ids', 'false_positive_ids']:
        for unit in result['scores'][kind]:
            decision = by_unit[unit]
            print(result['question_id'], kind, decision['pointer'])
            print(decision['original_statement'])
            print('Source-review reason:', decision['rationale'])


Additional v2 mistakes (original source text):
H01 false_negative_ids /statements/2
{'type': 'Precautions re: Pregnant or Nursing or Prescription Medications', 'notes': 'CAUTIONS/INTERACTIONS: Magnesium can interact with some antibiotics, diuretics, bisphosphonates, and muscle relaxants. In addition, many medications can affect magnesium absorption and excretion.'}
Source-review reason: Explicit captured precaution, warning, consultation requirement or usage restriction.
H04 false_negative_ids /statements/6
{'type': 'Precautions re: All Other', 'notes': 'Use only if safety seal is intact.'}
Source-review reason: Explicit captured precaution, warning, consultation requirement or usage restriction.
H04 false_positive_ids /statements/2
{'type': 'Precautions re: All Other', 'notes': 'Warning: See side panel.'}
Source-review reason: Warning locator only; retain optionally while the specific side-panel precautions are separate primary units.
H10 false_negative_ids /statements/17
{'type': 'Pr

### Findings and decision

| Set | v1 macro precision / recall | v2 macro precision / recall |
|---|---|---|
| Original 30 development questions | 98.4% / 96.5% | 100.0% / 99.8% |
| Twelve additional diagnostic cases | 97.2% / 97.5% | 97.8% / 83.0% |

**The additional-set precision averages have different denominators (v1: 10; v2: 9)** because v2 returns no primary evidence for a case that has a relevant warning. Both recall averages use 10 cases. The recall drop and raw false-negative counts are the important finding; do not read the precision increase as an overall improvement.

- V2 fixes most original errors, but incorrectly demotes a Vitamin D dosing-error caution when structured numeric values are null. Its full source text contains the relevant information.
- New wording missed by v2 includes a `CAUTIONS/INTERACTIONS` statement, “Use only if safety seal is intact,” a “Check with a qualified health care professional” instruction, and a loading-phase restriction embedded in directions.
- V2 also treats “Warning: See side panel” as primary evidence rather than optional locating context.
- Primary false negatives on the new cases rise from **1 to 4**. All structured source/mapping/context/abstention checks still pass, showing why those checks cannot substitute for relevance evaluation.

**Decision: do not replace v1 with v2.** Preserve v2 as an evaluated candidate; the graph, prior baseline and frozen releases are unchanged. No embeddings, model training or clinical approval were introduced.

**Next:** develop a conservative combined classifier: retain source precautions unless there is an explicit non-warning reason, add content-based restrictions from other categories, and avoid treating null numeric fields as proof that a dose statement is merely context. Evaluate that new revision on another newly selected set; do not claim this inspected diagnostic set remains unseen.


## Conservative relevance revision v3 — second diagnostic round

- Retain source-typed precautions unless clearly outside scope; find additional restrictions in other statement fields. Keep warning locators as context.
- Missing numeric dose values alone no longer demote a NatMed dose annotation. Original excerpts, provenance and pending clinical review remain intact.
- v3 was locked before selecting six further warning cases. The original 30 and first additional 12 cases are now development data.
- Six newly annotated cases exclude labels and collected source families in the earlier evaluation sets. They are still from the same collection, with assistant-reviewed relevance labels pending user adjudication; they are not an independent benchmark.
- v3 is the preferred experimental revision. v1/v2 scripts and results remain preserved; callers must explicitly select `kg_retrieval_v3.retrieve`. No raw data or graph changes.


In [16]:
from pathlib import Path
import json, hashlib
v3_root = Path.cwd()
if not (v3_root / 'data').is_dir():
    v3_root = v3_root.parent
v3_folder = v3_root / 'data/evaluation/retrieval_v3_comparison'
v3_manifest = json.loads((v3_folder / 'manifest.json').read_text())
for category in ['inputs', 'code', 'files']:
    for name, digest in v3_manifest[category].items():
        base = v3_folder if category == 'files' else v3_root
        assert hashlib.sha256((base / name).read_bytes()).hexdigest() == digest
v3_report = json.loads((v3_folder / 'comparison.json').read_text())
print('Verified saved comparison inputs, code and outputs.')
print(f'{"Set / version":42} {"Precision":>10} {"Recall":>10} {"P/R n":>8} {"FP/FN":>8}')
for name, versions in v3_report['sets'].items():
    for version, score in versions.items():
        print(f'{name + " / " + version:42} {score["macro_precision"]:10.1%} {score["macro_recall"]:10.1%} {str(score["precision_denominator"]) + "/" + str(score["recall_denominator"]):>8} {str(score["total_false_positives"]) + "/" + str(score["total_false_negatives"]):>8}')
print('Second diagnostic exact-text overlap:', v3_report['second_diagnostic_overlap'])


Verified saved comparison inputs, code and outputs.
Set / version                               Precision     Recall    P/R n    FP/FN
original_development / v1                       98.4%      96.5%    29/29      6/7
original_development / v2                      100.0%      99.8%    29/29      0/1
original_development / v3                      100.0%     100.0%    29/29      0/0
first_diagnostic_now_development / v1           97.2%      97.5%    10/10      2/1
first_diagnostic_now_development / v2           97.8%      83.0%     9/10      1/4
first_diagnostic_now_development / v3          100.0%     100.0%    10/10      0/0
second_diagnostic / v1                         100.0%      84.3%      5/5      0/3
second_diagnostic / v2                         100.0%      85.3%      5/5      0/3
second_diagnostic / v3                         100.0%      89.3%      5/5      0/2
Second diagnostic exact-text overlap: {'warning_occurrences_exact_text_overlap': 7, 'warning_occurrences_novel_text': 

In [17]:
v3_cases = json.loads((v3_root / 'data/evaluation/additional_evaluation_cases_v2.json').read_text())
for result in v3_report['sets']['second_diagnostic']['v3']['results']:
    missed = set(result['scores']['false_negative_ids'])
    for decision in v3_cases['decisions']:
        if decision['question_id'] == result['question_id'] and decision['unit_id'] in missed:
            print(result['question_id'], decision['source_file'], decision['pointer'])
            print(decision['original_statement']['notes'])
            print()


J01 data/raw/dsld/labels/label_1452.json /statements/12
DIRECTIONS: For adults, take one (1) to five (5) capsules daily, preferably with a meal.

J04 data/raw/dsld/labels/label_229239.json /statements/5
Directions: Adults: place 1 teaspoon (5 mL) into 6-8 oz of water or juice as maintenance. Use more if needed or as recommended by your healthcare professional.



### Findings and remaining work

- **Development:** v3 reaches 100% macro precision/recall on both the original 30 and first 12 additional cases. These informed its rules, so this is regression coverage, not generalization evidence.
- **Second diagnostic (6 cases):** 100% macro precision, **89.3% macro recall**, compared with v1's 100% / 84.3%. Five cases have defined metric denominators; J03 has no captured warning and contributes no precision/recall score. Absence of a warning is not proof of safety.
- **Two missed units:** adult-use directions in J01 and J04. No false positives or evidence-contract violations were observed. The locked rules were not tuned against these six cases.
- **Overlap:** 7 primary warning occurrences repeat earlier exact text; 16 have novel text. Deduplicated evidence units, rather than raw occurrences, are scored.
- v2's recall regression on the first diagnostic set remains documented. v3 recovers those misses using a conservative source-category fallback.
- Next: review the relevance labels and adult-use scope, then address remaining wording coverage and evaluate on further independently reviewed examples. These scores evaluate evidence retrieval with known IDs/categories, not natural-language answers or clinical correctness.
